# Day 40：LangChain Agent 与工具（Jupyter Notebook 实战手册）

> **所属阶段**：第 4 阶段 · 第四阶段：Agent 智能体开发与多 Agent 协同 ｜ **第 6 周**：Agent 基础、工具调用与 LangGraph 状态机  
> **GitLab 工单**：`STARMIND-040` ｜ **开发分支**：`feat/day40-tool-calling-executor`  
> **配套 HTML5 课件**：`courseware/day40/index.html` ｜ **独立版本快照**：`versions/day40/`

---

## 🎙️ 老陈开场旁白

各位早！昨天大家是不是体会到正则抠 `Action Input` 的别扭了？要是工具需要传 `filename`、`content`、`overwrite` 三个参数，靠字符串拼接迟早炸锅。今天老陈带大家拆解 LangChain 工具体系最优雅的设计：一个 `@tool` 装饰器怎么利用 Python 类型注解和 Pydantic 自动生成 OpenAI/Qwen 标准 JSON Schema，以及 `ToolCallingAgent` 和 `AgentExecutor` 是怎么通过 Callback 机制让你把中间每一步看得一清二楚的。

### 承上启下
- **昨日基线**：昨天（Day 39）我们零框架手写了基于文本正则解析的 ReAct Agent，体验了“思考-行动-观察”闭环，同时也暴露了单字符串 `Action Input` 难以传递多参数、缺乏类型强校验的痛点。
- **明日预告**：明天（Day 41）我们正式进入本阶段最核心的框架——LangGraph 状态图。你会发现 `AgentExecutor` 黑盒循环在复杂业务（例如“先审核再发邮件”）面前力不从心，而状态机图编排将彻底解放控制权。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day40'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-040）

星辰数科研发效能组希望星辰智脑不仅能回答问题，还能帮研发同学检索文档、执行受控的数据计算脚本，并在指定工作区自动生成周报 Markdown 文件。

**今日迭代目标：**
- 实现与 LangChain `@tool` 原理 1:1 对齐的装饰器，自动从函数签名、类型注解与 docstring 生成 JSON Schema 与 Pydantic 校验模型。
- 开发企业级安全三件套工具：内网搜索、AST 沙箱计算器、受限沙箱目录文件读写工具。
- 实现 `ToolCallingAgent`、`AgentExecutor` 与 `CallbackHandler` 观测机制，完整捕获 `intermediate_steps`。

**验收标准：**
- `@tool` 装饰任意带类型注解的函数后，可导出标准 OpenAI `function` JSON Schema，且调用前自动触发 Pydantic 类型校验。
- 文件读写工具严格拦截任何试图逃逸沙箱目录的相对/绝对路径，代码执行工具拦截危险模块导入。
- `AgentExecutor` 执行结束后返回 `output` 和完整的 `intermediate_steps`，且所有回调事件按序触发。

---

## 📚 课堂核心技术笔记

### 1. 1. 真实 LangChain 生产写法对照（langchain >= 0.3）

在生产环境中安装 `langchain` 与 `langchain-openai` 后，核心骨架与我们今天手写的类完全一致：
```python
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_tool_calling_agent, AgentExecutor

@tool
def calc(expr: str) -> str:
    '''安全计算数学表达式'''
    return safe_calculator(expr)

llm = ChatOpenAI(model='qwen-plus', base_url='https://dashscope.aliyuncs.com/compatible-mode/v1')
agent = create_tool_calling_agent(llm, [calc], prompt)
executor = AgentExecutor(agent=agent, tools=[calc], return_intermediate_steps=True, max_iterations=6)
```
理解了今天的源码实现，你就能秒懂 `create_tool_calling_agent` 背后到底干了什么。

> **⚠️ 老陈避坑**：坑点：函数 docstring 绝不能空着！LangChain 的 `@tool` 强制要求 docstring 作为工具描述，如果没有写 docstring 会直接抛出 `ValueError`。

### 2. 2. Pydantic 参数校验如何防止“工具参数幻觉”

即使使用了支持 Function Calling 的大模型，偶尔也会漏传必填参数、把字典传成字符串、或者传入超出业务范围的数值（如 `limit=-5`）。给工具绑定 Pydantic `BaseModel` 后，可以在真正执行业务逻辑前拦截一切脏参数，并将 `ValidationError` 的精简字段错误返回给模型，让模型在下一轮自动补全正确参数。

> **⚠️ 老陈避坑**：坑点：不要把冗长的 Pydantic 堆栈全部塞回上下文，只提取 `err['loc']` 和 `err['msg']` 即可，既省 Token 又让模型更容易看懂错在哪。

### 3. 3. Callback 回调体系的价值：流式前端与监控埋点

用户问一个复杂问题，Agent 可能要跑 8 秒。如果前端干等 8 秒才出字，用户会以为页面卡死了。通过 `CallbackHandler` 监听 `on_tool_start` 和 `on_tool_end`，后端可以通过 SSE 实时往前端推送“正在检索星辰知识库... -> 正在生成周报文件...”，体感延迟直接减半。

> **⚠️ 老陈避坑**：坑点：Callback 里的逻辑必须轻量且做好 `try/except`，绝不能因为打日志失败把正常的 Agent 执行流程搞崩。


## 💻 实操步骤 1：步骤一：手写 @tool 装饰器与企业安全工具三件套

- **目标文件**：`versions/day40/src/day40_tool_schema.py`
- **老陈代码旁白**：先来复刻 LangChain 最经典的 `@tool` 装饰器！利用 `inspect` 和 `pydantic.create_model`，只要给普通 Python 函数加上 `@tool`，它就会摇身一变成为带 JSON Schema 导出和自动参数校验的 `StructuredTool` 对象。接着我们用它封装搜索、安全代码求值、沙箱文件读写三个工具。


In [ ]:
"""Day 40 - 步骤一：@tool 装饰器、JSON Schema 自动提取与安全工具三件套。"""

from __future__ import annotations

import ast
import inspect
import operator
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Type, get_type_hints

from pydantic import BaseModel, Field, ValidationError, create_model


class StructuredTool:
    """对齐 LangChain StructuredTool 核心设计：封装函数、Schema 与运行时校验。"""

    def __init__(
        self,
        name: str,
        description: str,
        func: Callable[..., Any],
        args_schema: Type[BaseModel],
    ) -> None:
        self.name = name
        self.description = description
        self.func = func
        self.args_schema = args_schema

    def to_openai_schema(self) -> Dict[str, Any]:
        """导出符合 OpenAI / Qwen Function Calling 规范的 JSON Schema。"""
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": self.args_schema.model_json_schema(),
            },
        }

    def invoke(self, raw_args: Dict[str, Any]) -> str:
        """先经 Pydantic 校验并转换参数类型，再安全执行底层函数。"""
        try:
            validated = self.args_schema(**raw_args)
        except ValidationError as exc:
            details = "; ".join(
                f"参数 {'.'.join(map(str, e['loc']))}: {e['msg']}" for e in exc.errors()
            )
            return f"参数校验失败：{details}"
        try:
            result = self.func(**validated.model_dump())
            return str(result)
        except Exception as exc:
            return f"工具执行异常：{exc}"


def tool(func: Callable[..., Any]) -> StructuredTool:
    """
    极简 @tool 装饰器：自动提取函数名、docstring、类型注解与默认值，
    构建 Pydantic 校验模型并包装为 StructuredTool。
    """
    doc = (inspect.getdoc(func) or "").strip()
    if not doc:
        raise ValueError(f"工具函数 '{func.__name__}' 必须编写 docstring 作为工具说明")

    sig = inspect.signature(func)
    type_hints = get_type_hints(func)
    fields: Dict[str, Any] = {}

    for param_name, param in sig.parameters.items():
        annotation = type_hints.get(param_name, str)
        if param.default is inspect.Parameter.empty:
            fields[param_name] = (annotation, Field(..., description=f"参数 {param_name}"))
        else:
            fields[param_name] = (annotation, Field(default=param.default, description=f"参数 {param_name}"))

    args_model = create_model(f"{func.__name__.title()}Args", **fields)
    return StructuredTool(
        name=func.__name__,
        description=doc,
        func=func,
        args_schema=args_model,
    )


# ----------------- 企业级安全工具三件套 -----------------

_SAFE_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
}


def _safe_eval_expr(expression: str) -> float:
    def _walk(node: ast.AST) -> float:
        if isinstance(node, ast.Expression):
            return _walk(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return float(node.value)
        if isinstance(node, ast.BinOp) and type(node.op) in _SAFE_OPS:
            left, right = _walk(node.left), _walk(node.right)
            if isinstance(node.op, ast.Pow) and abs(right) > 50:
                raise ValueError("指数超过安全上限 50")
            return _SAFE_OPS[type(node.op)](left, right)
        raise ValueError(f"禁止执行的语法节点: {type(node).__name__}")

    return _walk(ast.parse(expression.strip(), mode="eval"))


def build_enterprise_tools(workspace_dir: Path) -> List[StructuredTool]:
    """构建绑定到指定安全沙箱目录的企业工具集。"""
    base_dir = workspace_dir.resolve()
    base_dir.mkdir(parents=True, exist_ok=True)

    docs_db = [
        {"id": "DOC-01", "topic": "星辰数科Q3研发目标", "content": "Q3目标：星辰智脑日活突破5000，平均响应延迟低于1.8秒。"},
        {"id": "DOC-02", "topic": "发布上线审批规范", "content": "生产环境发布需至少1名架构师Approved且单测覆盖率不低于80%。"},
    ]

    @tool
    def knowledge_search(keyword: str, top_k: int = 2) -> str:
        """检索星辰数科内部研发文档与规范，支持指定返回条数 top_k。"""
        hits = [d for d in docs_db if keyword in d["topic"] or keyword in d["content"]]
        if not hits:
            return f"未检索到包含 '{keyword}' 的文档。"
        return "\n".join(f"[{d['id']}] {d['topic']}: {d['content']}" for d in hits[:top_k])

    @tool
    def sandbox_python_calc(expression: str) -> str:
        """在 AST 安全沙箱中执行数学表达式计算，严禁函数调用与系统命令。"""
        val = _safe_eval_expr(expression)
        return str(int(val)) if val.is_integer() else f"{val:.2f}"

    @tool
    def safe_write_file(filename: str, content: str) -> str:
        """在受限工作区目录内写入文本文件，自动防御路径穿越攻击。"""
        target = (base_dir / filename).resolve()
        if not target.is_relative_to(base_dir):
            raise PermissionError(f"安全拦截：禁止访问工作区外路径 '{filename}'")
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding="utf-8")
        return f"已安全写入文件: {target.relative_to(base_dir)} ({len(content)} 字符)"

    @tool
    def safe_read_file(filename: str) -> str:
        """在受限工作区目录内读取文本文件内容，自动防御路径穿越攻击。"""
        target = (base_dir / filename).resolve()
        if not target.is_relative_to(base_dir):
            raise PermissionError(f"安全拦截：禁止读取工作区外路径 '{filename}'")
        if not target.exists():
            return f"文件不存在: {filename}"
        return target.read_text(encoding="utf-8")

    return [knowledge_search, sandbox_python_calc, safe_write_file, safe_read_file]


if __name__ == "__main__":
    import tempfile
    import json

    with tempfile.TemporaryDirectory() as tmp:
        tools = build_enterprise_tools(Path(tmp))
        print("=== 自动提取的 JSON Schema 示例 ===")
        print(json.dumps(tools[0].to_openai_schema(), ensure_ascii=False, indent=2))
        print("安全写入:", tools[2].invoke({"filename": "weekly.md", "content": "# 本周进展\n完成工具体系"}))
        print("路径穿越拦截:", tools[3].invoke({"filename": "../../etc/passwd"}))


### 🔍 步骤 1 代码深度解析

1. `@tool` 装饰器借助 `get_type_hints` 与 `pydantic.create_model` 动态构造强类型参数模型，完全复现了 LangChain 的核心黑魔法。
2. `safe_write_file` 和 `safe_read_file` 使用 `target.resolve().is_relative_to(base_dir)` 校验路径归属，哪怕传入 `sub/../../../etc/passwd` 也会在规范化后被精准拦下。
3. `StructuredTool.invoke` 将参数校验错误与业务执行异常都收敛为可读字符串返回，保证 Agent 调度器平稳运行。

**预期运行输出参考：**
```text
=== 自动提取的 JSON Schema 示例 ===
{
  "type": "function",
  "function": {
    "name": "knowledge_search",
    "description": "检索星辰数科内部研发文档与规范，支持指定返回条数 top_k。",
    "parameters": {
      "properties": {
        "keyword": {
          "description": "参数 keyword",
          "title": "Keyword",
          "type": "string"
        },
        "top_k": {
          "default": 2,
          "description": "参数 top_k",
          "title": "Top K",
          "type": "integer"
        }
      },
      "required": [
        "keyword"
      ],
      "title": "Knowledge_SearchArgs",
      "type": "object"
    }
  }
}
安全写入: 已安全写入文件: weekly.md (13 字符)
路径穿越拦截: 工具执行异常：安全拦截：禁止读取工作区外路径 '../../etc/passwd'
```


## 💻 实操步骤 2：步骤二：实现 ToolCallingAgent、AgentExecutor 与回调观测体系

- **目标文件**：`versions/day40/src/day40_agent_executor.py`
- **老陈代码旁白**：有了结构化工具，现在来搭建 `ToolCallingAgent` 和 `AgentExecutor`。特别留意 `CallbackHandler` 的设计——在生产项目里，我们正是靠这套回调钩子把 Agent 的每一次决策和工具耗时接入 Prometheus 和前端进度条的。


In [ ]:
"""Day 40 - 步骤二：ToolCallingAgent、AgentExecutor 与 CallbackHandler 观测引擎。"""

from __future__ import annotations

import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

try:
    from src.day40_tool_schema import StructuredTool, build_enterprise_tools
except ImportError:
    from day40_tool_schema import StructuredTool, build_enterprise_tools


@dataclass
class ToolCallAction:
    """模型输出的结构化工具调用指令（对齐 OpenAI tool_calls）。"""

    tool_name: str
    tool_args: Dict[str, Any]
    thought: str = ""


@dataclass
class AgentFinish:
    """模型输出的最终完成信号。"""

    output: str


class CallbackHandler:
    """可插拔回调基类：用于监听 Agent 全生命周期事件。"""

    def on_agent_action(self, action: ToolCallAction) -> None:
        pass

    def on_tool_start(self, tool_name: str, tool_args: Dict[str, Any]) -> None:
        pass

    def on_tool_end(self, tool_name: str, observation: str, latency_ms: float) -> None:
        pass

    def on_agent_finish(self, finish: AgentFinish) -> None:
        pass


class TraceCallbackHandler(CallbackHandler):
    """实战观测回调器：收集结构化事件日志供排查与前端进度展示。"""

    def __init__(self) -> None:
        self.events: List[Dict[str, Any]] = []

    def on_agent_action(self, action: ToolCallAction) -> None:
        self.events.append({"event": "agent_action", "tool": action.tool_name, "args": action.tool_args})

    def on_tool_start(self, tool_name: str, tool_args: Dict[str, Any]) -> None:
        self.events.append({"event": "tool_start", "tool": tool_name})

    def on_tool_end(self, tool_name: str, observation: str, latency_ms: float) -> None:
        self.events.append(
            {"event": "tool_end", "tool": tool_name, "latency_ms": latency_ms, "preview": observation[:60]}
        )

    def on_agent_finish(self, finish: AgentFinish) -> None:
        self.events.append({"event": "agent_finish", "output_len": len(finish.output)})


class MockToolCallingAgent:
    """
    本地确定性 ToolCallingAgent（模拟原生 Function Calling 模型行为）。
    接收用户输入与历史中间步骤，决定返回 ToolCallAction 或 AgentFinish。
    """

    def plan(
        self,
        user_input: str,
        intermediate_steps: List[Tuple[ToolCallAction, str]],
        tool_schemas: List[Dict[str, Any]],
    ) -> ToolCallAction | AgentFinish:
        assert len(tool_schemas) > 0, "必须向模型提供工具 Schema 列表"

        # 场景：查 Q3 目标并写入周报文件
        if "Q3" in user_input and "写入" in user_input:
            if len(intermediate_steps) == 0:
                return ToolCallAction(
                    tool_name="knowledge_search",
                    tool_args={"keyword": "Q3", "top_k": 1},
                    thought="先检索星辰数科 Q3 研发目标文档",
                )
            if len(intermediate_steps) == 1:
                doc_obs = intermediate_steps[0][1]
                return ToolCallAction(
                    tool_name="safe_write_file",
                    tool_args={"filename": "q3_summary.md", "content": f"# Q3目标摘录\n{doc_obs}\n"},
                    thought="已获取目标内容，将其安全写入 q3_summary.md",
                )
            write_obs = intermediate_steps[-1][1]
            return AgentFinish(output=f"已完成检索并生成文件：{write_obs}")

        return AgentFinish(output="未匹配到需要调用工具的复合指令，直接回复完成。")


class AgentExecutor:
    """对齐 LangChain AgentExecutor：负责驱动循环、工具分发、熔断与回调通知。"""

    def __init__(
        self,
        agent: MockToolCallingAgent,
        tools: List[StructuredTool],
        callbacks: Optional[List[CallbackHandler]] = None,
        max_iterations: int = 5,
    ) -> None:
        self.agent = agent
        self.tool_map: Dict[str, StructuredTool] = {t.name: t for t in tools}
        self.tool_schemas: List[Dict[str, Any]] = [t.to_openai_schema() for t in tools]
        self.callbacks = callbacks or []
        self.max_iterations = max_iterations

    def invoke(self, user_input: str) -> Dict[str, Any]:
        intermediate_steps: List[Tuple[ToolCallAction, str]] = []

        for _ in range(self.max_iterations):
            decision = self.agent.plan(user_input, intermediate_steps, self.tool_schemas)
            if isinstance(decision, AgentFinish):
                for cb in self.callbacks:
                    cb.on_agent_finish(decision)
                return {
                    "input": user_input,
                    "output": decision.output,
                    "intermediate_steps": intermediate_steps,
                }

            for cb in self.callbacks:
                cb.on_agent_action(decision)

            tool_obj = self.tool_map.get(decision.tool_name)
            t0 = time.perf_counter()
            for cb in self.callbacks:
                cb.on_tool_start(decision.tool_name, decision.tool_args)

            if tool_obj is None:
                observation = f"错误：未注册工具 '{decision.tool_name}'"
            else:
                observation = tool_obj.invoke(decision.tool_args)

            latency_ms = round((time.perf_counter() - t0) * 1000, 3)
            for cb in self.callbacks:
                cb.on_tool_end(decision.tool_name, observation, latency_ms)

            intermediate_steps.append((decision, observation))

        return {
            "input": user_input,
            "output": f"已达到最大迭代次数 {self.max_iterations}，任务中止。",
            "intermediate_steps": intermediate_steps,
        }


if __name__ == "__main__":
    import tempfile

    with tempfile.TemporaryDirectory() as tmp:
        tools = build_enterprise_tools(Path(tmp))
        tracer = TraceCallbackHandler()
        executor = AgentExecutor(agent=MockToolCallingAgent(), tools=tools, callbacks=[tracer])
        res = executor.invoke("帮我查一下公司Q3研发目标，并写入周报文件")
        print("最终输出:", res["output"])
        print("中间步骤数:", len(res["intermediate_steps"]))
        print("回调事件流:", [e["event"] for e in tracer.events])


### 🔍 步骤 2 代码深度解析

1. `AgentExecutor.invoke` 返回的字典结构（`input`/`output`/`intermediate_steps`）与 LangChain `AgentExecutor` 保持完全一致。
2. `TraceCallbackHandler` 以非侵入方式捕获了每次 `agent_action -> tool_start -> tool_end -> agent_finish` 的完整时间线。
3. 工具分发时直接调用 `tool_obj.invoke(decision.tool_args)`，享受前一步 Pydantic 强类型校验的安全兜底。

**预期运行输出参考：**
```text
最终输出: 已完成检索并生成文件：已安全写入文件: q3_summary.md (48 字符)
中间步骤数: 2
回调事件流: ['agent_action', 'tool_start', 'tool_end', 'agent_action', 'tool_start', 'tool_end', 'agent_finish']
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day40/tests/test_day40.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 40 自动化单元测试：验证 @tool Schema 提取、安全工具防御与 AgentExecutor 回调链。"""

from pathlib import Path
import pytest
from src.day40_tool_schema import tool, build_enterprise_tools
from src.day40_agent_executor import (
    AgentExecutor,
    MockToolCallingAgent,
    TraceCallbackHandler,
)


def test_tool_decorator_schema_and_validation():
    @tool
    def multiply(a: int, b: int = 2) -> int:
        """计算两个整数的乘积。"""
        return a * b

    schema = multiply.to_openai_schema()
    assert schema["function"]["name"] == "multiply"
    assert "a" in schema["function"]["parameters"]["required"]
    assert "b" not in schema["function"]["parameters"]["required"]
    # 自动类型转换: '5' -> 5
    assert multiply.invoke({"a": "5", "b": 3}) == "15"
    # 非法类型拦截
    assert "参数校验失败" in multiply.invoke({"a": "not_an_int"})


def test_enterprise_tools_security(tmp_path: Path):
    tools = {t.name: t for t in build_enterprise_tools(tmp_path)}
    assert "5000" in tools["knowledge_search"].invoke({"keyword": "Q3"})
    assert tools["sandbox_python_calc"].invoke({"expression": "(10 + 5) * 4"}) == "60"
    assert "禁止执行" in tools["sandbox_python_calc"].invoke({"expression": "open('/etc/passwd')"})
    assert "安全拦截" in tools["safe_write_file"].invoke({"filename": "../escape.txt", "content": "x"})
    assert "安全拦截" in tools["safe_read_file"].invoke({"filename": "/etc/hosts"})


def test_agent_executor_full_workflow_and_callbacks(tmp_path: Path):
    tools = build_enterprise_tools(tmp_path)
    tracer = TraceCallbackHandler()
    executor = AgentExecutor(agent=MockToolCallingAgent(), tools=tools, callbacks=[tracer])

    res = executor.invoke("查询公司Q3研发目标并写入文件")
    assert "q3_summary.md" in res["output"]
    assert len(res["intermediate_steps"]) == 2
    assert (tmp_path / "q3_summary.md").exists()
    assert "5000" in (tmp_path / "q3_summary.md").read_text(encoding="utf-8")

    event_types = [e["event"] for e in tracer.events]
    assert event_types[0] == "agent_action"
    assert event_types[-1] == "agent_finish"
    assert event_types.count("tool_end") == 2

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：为文件工具增加扩展名白名单与大小限制

**题目要求：**
编写一个带 `@tool` 装饰的 `safe_export_doc` 工具，只允许写入 `.md`、`.txt`、`.json` 三种后缀文件，且内容长度不得超过 10,000 字符。

**思路提示：** 使用 `Path(filename).suffix.lower() in {'.md', '.txt', '.json'}` 校验后缀，校验 `len(content) <= 10000`。


In [ ]:
"""Day 40 作业 1 参考答案：扩展名白名单与文件大小护栏。"""

from pathlib import Path
import tempfile

ALLOWED_EXTS = {".md", ".txt", ".json"}
MAX_CHARS = 10_000


def safe_export_doc(base_dir: Path, filename: str, content: str) -> str:
    """受控导出文档：限定扩展名、大小与目录边界。"""
    ext = Path(filename).suffix.lower()
    if ext not in ALLOWED_EXTS:
        return f"拒绝写入：不支持的文件类型 '{ext}'，仅允许 {sorted(ALLOWED_EXTS)}"
    if len(content) > MAX_CHARS:
        return f"拒绝写入：内容长度 {len(content)} 超过上限 {MAX_CHARS}"
    target = (base_dir / filename).resolve()
    if not target.is_relative_to(base_dir.resolve()):
        return "拒绝写入：检测到路径穿越风险"
    target.write_text(content, encoding="utf-8")
    return f"导出成功：{target.name}"


if __name__ == "__main__":
    with tempfile.TemporaryDirectory() as tmp:
        p = Path(tmp)
        assert "拒绝写入" in safe_export_doc(p, "hack.sh", "rm -rf /")
        assert "导出成功" in safe_export_doc(p, "report.md", "# OK")
        print("作业 1 验证通过")


**答案解析与评分标准：**

企业文件安全永远是“目录边界 + 扩展名白名单 + 大小配额”三道防线，缺一不可。


### 作业 2（企业实战）：编写工具执行超时保护装饰器

**题目要求：**
外部搜索或 API 工具偶发卡死，请使用 `concurrent.futures.ThreadPoolExecutor` 实现 `run_with_timeout(func, args, timeout_sec)`，超时后立即返回可读的超时提示给 Agent。

**思路提示：** 提交任务后调用 `future.result(timeout=timeout_sec)` 并捕获 `TimeoutError`。


In [ ]:
"""Day 40 作业 2 参考答案：工具超时隔离执行器。"""

import time
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout
from typing import Any, Callable, Dict


def run_with_timeout(func: Callable[..., Any], kwargs: Dict[str, Any], timeout_sec: float = 1.0) -> str:
    pool = ThreadPoolExecutor(max_workers=1)
    future = pool.submit(func, **kwargs)
    try:
        return str(future.result(timeout=timeout_sec))
    except FuturesTimeout:
        return f"工具执行超时（>{timeout_sec}s），请降级处理或跳过该工具。"
    finally:
        pool.shutdown(wait=False, cancel_futures=True)


if __name__ == "__main__":
    def slow_api(q: str) -> str:
        time.sleep(0.5)
        return q

    res_timeout = run_with_timeout(slow_api, {"q": "hello"}, timeout_sec=0.1)
    res_ok = run_with_timeout(lambda q: f"ok:{q}", {"q": "fast"}, timeout_sec=1.0)
    print(res_timeout, "|", res_ok)
    assert "超时" in res_timeout and res_ok == "ok:fast"


**答案解析与评分标准：**

注意 `finally` 中使用 `pool.shutdown(wait=False)` 避免主线程被卡死的子线程阻塞，这是高并发 Agent 服务保命的常见手法。


### 作业 3（架构进阶）：实现动态工具权限过滤（RBAC）

**题目要求：**
在星辰数科内部，实习生只能调用 `knowledge_search`，正式员工可调用全部工具。请设计 `ToolPermissionFilter`，按用户角色过滤暴露给 LLM 的 `tools` 列表并在调用期二次鉴权。

**思路提示：** 维护 `role -> allowed_tool_names` 映射表，既在 Schema 暴露阶段裁剪，也在 `invoke` 前做强制检查（防止提示词注入绕过）。


In [ ]:
"""Day 40 作业 3 参考答案：基于 RBAC 的工具动态裁剪与二次鉴权。"""

from typing import Dict, List, Set


class ToolPermissionFilter:
    ROLE_PERMISSIONS: Dict[str, Set[str]] = {
        "intern": {"knowledge_search"},
        "engineer": {"knowledge_search", "sandbox_python_calc", "safe_read_file", "safe_write_file"},
    }

    def filter_tool_names(self, role: str, all_tools: List[str]) -> List[str]:
        allowed = self.ROLE_PERMISSIONS.get(role, set())
        return [name for name in all_tools if name in allowed]

    def authorize(self, role: str, tool_name: str) -> bool:
        return tool_name in self.ROLE_PERMISSIONS.get(role, set())


if __name__ == "__main__":
    f = ToolPermissionFilter()
    all_t = ["knowledge_search", "safe_write_file"]
    assert f.filter_tool_names("intern", all_t) == ["knowledge_search"]
    assert f.authorize("intern", "safe_write_file") is False
    assert f.authorize("engineer", "safe_write_file") is True
    print("RBAC 工具鉴权验证通过")


**答案解析与评分标准：**

千万不能只在 Prompt 里告诉模型“实习生不能写文件”，必须在工具列表裁剪 + 运行时代码鉴权两层同时拦截，这才是零信任架构。


---

## 🎯 老陈收尾复盘

今天我们不仅搞懂了 `@tool` 和 `AgentExecutor` 的底层源码逻辑，还亲手给文件读写和代码执行工具套上了“路径穿越防护 + AST 沙箱”两把安全锁。记住：在企业里给 Agent 开放文件读写权限，如果不做 `resolve().is_relative_to(BASE_DIR)` 校验，安全团队半小时就会找你喝茶！
